# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [120]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [121]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head(20)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True


In [122]:
spaceship["HomePlanet"] = spaceship["HomePlanet"].map({
    "Europa": 0,
    "Asia": 1,
    "America": 2
})

In [123]:
spaceship["Cabin"] = spaceship["Cabin"].map({"S":0, "C":1, "Q":2})

**Check the shape of your data**

In [124]:
#shape
spaceship.shape 


(8693, 14)

**Check for data types**

In [125]:
#your code here
spaceship.dtypes

PassengerId      object
HomePlanet      float64
CryoSleep        object
Cabin           float64
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [126]:
#your code here
spaceship.isna().sum()

PassengerId        0
HomePlanet      6562
CryoSleep        217
Cabin           8693
Destination      182
Age              179
VIP              203
RoomService      181
FoodCourt        183
ShoppingMall     208
Spa              183
VRDeck           188
Name             200
Transported        0
dtype: int64

In [127]:
spaceship.dropna(subset=["Transported"], inplace=True)

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [128]:
print(spaceship["Cabin"].head(10))
print(spaceship["Cabin"].dtype)

cabin_parts = spaceship["Cabin"].astype("string").str.split("/", expand=True)
print(cabin_parts.shape)
print(cabin_parts.head())

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: Cabin, dtype: float64
float64
(8693, 1)
      0
0  <NA>
1  <NA>
2  <NA>
3  <NA>
4  <NA>


- Drop PassengerId and Name

In [129]:
#drop passanger id and name 
spaceship.drop(columns=["PassengerId", "Name"], inplace=True)
spaceship.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,0.0,False,NaN,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,NaN,False,NaN,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,0.0,False,NaN,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,0.0,False,NaN,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,NaN,False,NaN,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


- For non-numerical columns, do dummies.

In [130]:
#dummes for categorical variables
spaceship = pd.get_dummies(spaceship, drop_first=True)

**Perform Train Test Split**

In [131]:
#test train split
X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [132]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Split Cabin
cabin_parts = spaceship["Cabin"].fillna("Unknown/0/U").astype(str).str.split("/", expand=True)
spaceship["Cabin_deck"] = cabin_parts[0]
spaceship["Cabin_num"] = pd.to_numeric(cabin_parts[1], errors="coerce")
spaceship["Cabin_side"] = cabin_parts[2]

# Drop original Cabin
spaceship.drop(columns=["Cabin"], inplace=True, errors="ignore")

# Fill missing numeric values
numeric_cols = spaceship.select_dtypes(include=["number"]).columns
for col in numeric_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].median())

# Fill missing categorical values
categorical_cols = spaceship.select_dtypes(include=["object", "string", "bool"]).columns
for col in categorical_cols:
    spaceship[col] = spaceship[col].fillna("Unknown")

# Convert categorical variables
spaceship = pd.get_dummies(spaceship, drop_first=True)

# Define X and y
X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]

# If target is boolean, convert to 0/1
if y.dtype == "bool":
    y = y.astype(int)

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X, y)

y_pred = knn.predict(X)
print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred))

[[3468  847]
 [ 867 3511]]
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      4315
           1       0.81      0.80      0.80      4378

    accuracy                           0.80      8693
   macro avg       0.80      0.80      0.80      8693
weighted avg       0.80      0.80      0.80      8693



In [133]:
print(spaceship.shape)
print(spaceship.head())

(8693, 13)
   HomePlanet   Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  \
0         0.0  39.0          0.0        0.0           0.0     0.0     0.0   
1         0.0  24.0        109.0        9.0          25.0   549.0    44.0   
2         0.0  58.0         43.0     3576.0           0.0  6715.0    49.0   
3         0.0  33.0          0.0     1283.0         371.0  3329.0   193.0   
4         0.0  16.0        303.0       70.0         151.0   565.0     2.0   

   Transported  CryoSleep_True  Destination_PSO J318.5-22  \
0        False           False                      False   
1         True           False                      False   
2        False           False                      False   
3        False           False                      False   
4         True           False                      False   

   Destination_TRAPPIST-1e  VIP_True  Cabin_num  
0                     True     False          0  
1                     True     False          0  
2        

In [134]:
print(X.shape)
print(y.shape)

(8693, 12)
(8693,)


- Evaluate your model's performance. Comment it

In [ ]:
#model performance
from sklearn.metrics import classification_report, confusion_matrix 
y_pred = knn.predict(X)
print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred)) 


[[3468  847]
 [ 867 3511]]
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      4315
           1       0.81      0.80      0.80      4378

    accuracy                           0.80      8693
   macro avg       0.80      0.80      0.80      8693
weighted avg       0.80      0.80      0.80      8693



The KNN model achieved an overall accuracy of 80%, indicating that it correctly predicts the outcome for most observations in the dataset. The model shows balanced performance between the two classes, with precision and recall values around 0.80 for both class 0 and class 1.
#This suggests that the model is able to identify both transported and non-transported passengers with similar effectiveness, without strong bias toward one class. The F1-scores are also consistent at 0.80, confirming that the model maintains a good balance between precision and recall.